In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import torchvision.transforms as transforms

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
df = pd.read_csv("Data/train.csv")

y = df["label"].values
X = df.drop("label", axis=1).values

X = X / 255.0
X = X.reshape(-1, 1, 28, 28)

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# Original LeNet-5

In [4]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.AvgPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        self.fc1 = nn.Linear(16*4*4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = torch.tanh(self.pool(self.conv1(x)))
        x = torch.tanh(self.pool(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

# Modified LeNet-5 (ReLU + Dropout)

In [5]:
class LeNet5_Modified(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        
        self.fc1 = nn.Linear(16*4*4, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        x = F.relu(self.pool(self.conv1(x)))
        x = F.relu(self.pool(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# AlexNet (Adapted for MNIST)

In [6]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 192, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(192, 384, 3, padding=1),
            nn.ReLU(),
            
            nn.Conv2d(384, 256, 3, padding=1),
            nn.ReLU(),
            
            nn.Conv2d(256, 256, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256*3*3, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Training + Throughput + GPU Memory

In [7]:
def train_model(model, batch_size=64, epochs=5, lr=0.001):
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    
    start_time = time.time()
    total_images = 0
    
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            
            total_images += xb.size(0)
    
    end_time = time.time()
    
    throughput = total_images / (end_time - start_time)
    
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = torch.argmax(model(xb), 1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    
    accuracy = correct / total
    
    if device.type == "cuda":
        memory = torch.cuda.max_memory_allocated() / (1024**2)
    else:
        memory = "CPU"
    
    return accuracy, throughput, memory

# Batch Size Experiment

In [8]:
batch_sizes = [32, 64, 128, 256]

for bs in batch_sizes:
    acc, thr, mem = train_model(LeNet5_Modified(), batch_size=bs)
    
    print(f"\nBatch Size: {bs}")
    print("Accuracy:", acc)
    print("Throughput (images/sec):", thr)
    print("GPU Memory (MB):", mem)


Batch Size: 32
Accuracy: 0.983452380952381
Throughput (images/sec): 5541.30600410203
GPU Memory (MB): 20.6748046875

Batch Size: 64
Accuracy: 0.9769047619047619
Throughput (images/sec): 10095.252064511165
GPU Memory (MB): 23.818359375

Batch Size: 128
Accuracy: 0.9798809523809524
Throughput (images/sec): 16377.725451491253
GPU Memory (MB): 28.8876953125

Batch Size: 256
Accuracy: 0.9777380952380952
Throughput (images/sec): 24748.512809259355
GPU Memory (MB): 40.2138671875


- Larger batch → Higher throughput

- Larger batch → More GPU memory

- Very large batch may reduce generalization

# Does Dropout + ReLU Improve LeNet?

In [9]:
acc_original, _, _ = train_model(LeNet5(), epochs=5)
acc_modified, _, _ = train_model(LeNet5_Modified(), epochs=5)

print("Original LeNet Accuracy:", acc_original)
print("Modified LeNet Accuracy:", acc_modified)

Original LeNet Accuracy: 0.9754761904761905
Modified LeNet Accuracy: 0.9835714285714285


# Data Augmentation (Exploit Invariances)

In [10]:
augmentation = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.1,0.1))
])

# Increasing Epochs for AlexNet

In [11]:
acc_5, _, _ = train_model(AlexNet(), epochs=5)
acc_15, _, _ = train_model(AlexNet(), epochs=15)

print("AlexNet 5 epochs:", acc_5)
print("AlexNet 15 epochs:", acc_15)

AlexNet 5 epochs: 0.9885714285714285
AlexNet 15 epochs: 0.99


## 📌 Final Comparison & Explanation
Observed Behavior:

| Model   | Few Epochs         | More Epochs        |
| ------- | ------------------ | ------------------ |
| LeNet   | Quickly saturates  | Minor improvement  |
| AlexNet | Improves gradually | Larger improvement |
